# Isometries


In [1]:
from itertools import product

import sympy as sp

s2, s3, s6 = sp.sqrt(2), sp.sqrt(3), sp.sqrt(6)

# Encoder rows: |I1 I2 I3 R>; columns: |P>.
Venc = sp.Matrix([
    [0, 0],
    [0, 0],
    [0, 0],
    [-s3 / 3, 0],
    [sp.Rational(1, 2), 0],
    [s3 / 6, 0],
    [0, sp.Rational(1, 2)],
    [0, -s3 / 6],
    [sp.Rational(-1, 2), 0],
    [s3 / 6, 0],
    [0, sp.Rational(-1, 2)],
    [0, -s3 / 6],
    [0, 0],
    [0, s3 / 3],
    [0, 0],
    [0, 0],
])

# Decoder columns: |R O1 O2 O3>; rows: |F A1 A2 A3 A4>.
# Unlisted entries are zero.
Vdec = sp.SparseMatrix(32, 16, {
    (0, 3): s2 / 6,
    (0, 5): s2 / 6,
    (0, 6): -s2 / 3,
    (0, 11): s6 / 6,
    (0, 13): -s6 / 6,
    (1, 1): sp.Rational(-1, 3),
    (1, 2): sp.Rational(1, 6),
    (1, 4): sp.Rational(1, 6),
    (1, 10): -s3 / 6,
    (1, 12): s3 / 6,
    (2, 2): -s3 / 6,
    (2, 4): s3 / 6,
    (2, 9): -s2 / 3 - sp.Rational(1, 3),
    (2, 10): sp.Rational(1, 6) - s2 / 3,
    (2, 12): sp.Rational(1, 6) - s2 / 3,
    (3, 2): sp.Rational(1, 2),
    (3, 4): sp.Rational(-1, 2),
    (3, 9): -s3 / 3,
    (3, 10): s3 / 6,
    (3, 12): s3 / 6,
    (4, 3): sp.Rational(-1, 3) + s2 / 6,
    (4, 5): sp.Rational(-1, 3) + s2 / 6,
    (4, 6): -s2 / 3 - sp.Rational(1, 3),
    (4, 11): -s6 / 6,
    (4, 13): s6 / 6,
    (5, 0): 1,
    (6, 3): -s6 / 6,
    (6, 5): s6 / 6,
    (6, 11): sp.Rational(-1, 3) - s2 / 6,
    (6, 13): sp.Rational(-1, 3) - s2 / 6,
    (6, 14): sp.Rational(-1, 3) + s2 / 3,
    (7, 1): sp.Rational(1, 3) - s2 / 3,
    (7, 2): -s2 / 3 - sp.Rational(1, 6),
    (7, 4): -s2 / 3 - sp.Rational(1, 6),
    (7, 10): -s3 / 6,
    (7, 12): s3 / 6,
    (12, 8): -1,
    (17, 3): sp.Rational(1, 6),
    (17, 5): sp.Rational(1, 6),
    (17, 6): sp.Rational(-1, 3),
    (17, 11): s3 / 6,
    (17, 13): -s3 / 6,
    (18, 3): s3 / 6,
    (18, 5): -s3 / 6,
    (18, 11): sp.Rational(1, 6) - s2 / 3,
    (18, 13): sp.Rational(1, 6) - s2 / 3,
    (18, 14): -s2 / 3 - sp.Rational(1, 3),
    (19, 3): sp.Rational(1, 2),
    (19, 5): sp.Rational(-1, 2),
    (19, 11): -s3 / 6,
    (19, 13): -s3 / 6,
    (19, 14): s3 / 3,
    (20, 7): -1,
    (21, 1): sp.Rational(1, 3) + s2 / 3,
    (21, 2): sp.Rational(1, 3) - s2 / 6,
    (21, 4): sp.Rational(1, 3) - s2 / 6,
    (21, 10): -s6 / 6,
    (21, 12): s6 / 6,
    (22, 15): -1,
    (23, 3): -s2 / 3 - sp.Rational(1, 6),
    (23, 5): -s2 / 3 - sp.Rational(1, 6),
    (23, 6): sp.Rational(1, 3) - s2 / 3,
    (23, 11): s3 / 6,
    (23, 13): -s3 / 6,
    (24, 1): -s2 / 3,
    (24, 2): s2 / 6,
    (24, 4): s2 / 6,
    (24, 10): -s6 / 6,
    (24, 12): s6 / 6,
    (28, 2): s6 / 6,
    (28, 4): -s6 / 6,
    (28, 9): sp.Rational(-1, 3) + s2 / 3,
    (28, 10): sp.Rational(-1, 3) - s2 / 6,
    (28, 12): sp.Rational(-1, 3) - s2 / 6,
})

# |I1 I2 I3 R> -> |R I1 I2 I3>.
permutation = sp.zeros(16)
for slots, memory in product(range(8), range(2)):
    permutation[8 * memory + slots, 2 * slots + memory] = 1


## Fidelity

Apply the full depolarizing channel to each of the three physical slots;
the memory register is unchanged:

```math
\mathcal N_{\gamma,j}(A)=(1-\gamma)A+
\frac{\gamma}{4}(A+X_jAX_j+Y_jAY_j+Z_jAZ_j).
```

Encode each operator $|i\rangle\langle j|$, apply the noise, and decode.
Trace out the four ancillary outputs to obtain $\mathcal E_I(|i\rangle\langle j|)$,
then sum its $(i,j)$ elements:

```math
F_{\mathrm{ave}}(\mathcal N_\gamma,\mathcal C)
=\frac14\sum_{i,j=0}^{1}
\langle i|\mathcal E_I(|i\rangle\langle j|)|j\rangle.
```

The ancillary partial trace uses its computational basis:

```math
\operatorname{Tr}_{\mathrm{Anc}} B
=\sum_{a=0}^{15}(I_F\otimes\langle a|)B(I_F\otimes|a\rangle).
```


In [2]:
from IPython.display import Math, display

gamma = sp.symbols("gamma", real=True)
I2 = sp.eye(2)
X = sp.Matrix([[0, 1], [1, 0]])
Y = sp.Matrix([[0, -sp.I], [sp.I, 0]])
Z = sp.diag(1, -1)


def noise_on_slot(A, slot):
    # slot 0,1,2 means I1,I2,I3; the fourth factor is memory R.
    operators = []
    for P in (X, Y, Z):
        factors = [I2, I2, I2, I2]
        factors[slot] = P
        operators.append(sp.kronecker_product(*factors))
    Xj, Yj, Zj = operators
    depolarized = (A + Xj*A*Xj + Yj*A*Yj + Zj*A*Zj) / 4
    return ((1 - gamma)*A + gamma*depolarized).applyfunc(sp.expand)


def output_channel(Eij):
    # Encode in the order I1,I2,I3,R.
    A = Venc * Eij * Venc.H

    # Apply N_gamma to all three physical slots.
    for slot in range(3):
        A = noise_on_slot(A, slot)

    # Reorder to R,O1,O2,O3 and decode.
    A = permutation * A * permutation.H
    B = Vdec * A * Vdec.H

    # Tr_Anc(B) = sum_a (I_F tensor <a|) B (I_F tensor |a>).
    I_F = sp.eye(2)
    ancilla_basis = sp.eye(16)
    output = sp.zeros(2)
    for a in range(16):
        ket_a = ancilla_basis[:, a]  # |a>
        bra_a = ket_a.H             # <a|
        K_a = sp.kronecker_product(I_F, bra_a)
        output += K_a * B * K_a.H
    return output.applyfunc(sp.simplify)


F_ave = sp.S.Zero
for i, j in product(range(2), repeat=2):
    Eij = sp.zeros(2)
    Eij[i, j] = 1
    F_ave += output_channel(Eij)[i, j] / 4

F_ave = sp.collect(sp.expand(F_ave), gamma)
display(Math(r"F_{\mathrm{ave}}(\mathcal N_\gamma,\mathcal C) = "
             + sp.latex(F_ave)))


<IPython.core.display.Math object>

The calculated fidelity is rewritten in the form of Eq. (14) and checked for exact agreement.


In [3]:
numerator = sp.expand(24 * F_ave)
quotient, remainder = sp.div(numerator, 2 - gamma, gamma)

target = (2 - gamma) / 24 * (
    12
    + (8*sp.sqrt(2) - 9)*gamma
    + (3 - 8*sp.sqrt(2))*gamma**2
)
difference = sp.simplify(F_ave - target)

display(Math(r"24F_{\mathrm{ave}} = " + sp.latex(sp.collect(numerator, gamma))))
display(Math(r"\frac{24F_{\mathrm{ave}}}{2-\gamma} = " + sp.latex(quotient)))
display(Math(
    r"F_{\mathrm{ave}} = \frac{2-\gamma}{24}"
    r"\left[12+(8\sqrt{2}-9)\gamma+(3-8\sqrt{2})\gamma^2\right]"
))
display(Math(r"F_{\mathrm{ave}} - F_{\mathrm{text}} = " + sp.latex(difference)))

assert sp.simplify(remainder) == 0, "Polynomial division has a nonzero remainder."
assert difference == 0, "The notebook result differs from the manuscript formula."


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>